# Bug2Code — Final Colab Experiment Notebook

This notebook consolidates the commands used across the three Colab sessions into one chronological workflow.

The notebook keeps the original Colab-style commands (`!python`, `!git`, `%cd`, Google Drive mounting) rather than converting them into a Python orchestration script.

Because large artifacts and checkpoints are not stored in Git, some cells restore them from Google Drive.


## 1. Check GPU


In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


## 2. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Clone and install Bug2Code


In [ ]:
%cd /content
!git clone https://github.com/Shir-personal/Bug2Code.git
%cd /content/Bug2Code

!git checkout main
!git pull
!git log -1 --oneline
!pip install -e . --no-deps


## 4. Restore Cassandra experiment artifacts

These files were saved separately because the raw dataset, candidate caches, and model checkpoint are not committed to Git.


In [ ]:
!mkdir -p data/processed
!mkdir -p data/cache_cassandra30/candidates
!mkdir -p experiments_cassandra30/finetuned_codebert/checkpoints

!cp "/content/drive/MyDrive/Bug2Code_transfer_local/localization_dataset.parquet"     data/processed/localization_dataset.parquet

!cp "/content/drive/MyDrive/Bug2Code_transfer_local/cassandra30_subset.parquet"     data/processed/cassandra30_subset.parquet

!cp "/content/drive/MyDrive/Bug2Code_transfer_local/file_versions.parquet"     data/cache_cassandra30/candidates/file_versions.parquet

!cp "/content/drive/MyDrive/Bug2Code_transfer_local/snapshot_index.parquet"     data/cache_cassandra30/candidates/snapshot_index.parquet


## 5. Prepare Cassandra repository


In [ ]:
%cd /content/Bug2Code
!python -m bug2code.data.repos --config configs/cassandra30.yaml


## 6. Train Fine-tuned CodeBERT

A short benchmark was run first, followed by the full 3-epoch training run.


In [ ]:
!python -m bug2code.localization.train_codebert   --config configs/cassandra30.yaml   --benchmark-steps 15


In [ ]:
!python -m bug2code.localization.train_codebert   --config configs/cassandra30.yaml   --target-epoch 3


### Save trained checkpoints to Drive


In [ ]:
!mkdir -p /content/drive/MyDrive/Bug2Code_results/experiments_cassandra30/checkpoints

!cp experiments_cassandra30/finetuned_codebert/checkpoints/*.pt   /content/drive/MyDrive/Bug2Code_results/experiments_cassandra30/checkpoints/

!ls -lh /content/drive/MyDrive/Bug2Code_results/experiments_cassandra30/checkpoints/


## 7. Validation — select the Fine-tuned CodeBERT epoch


In [ ]:
!python -m bug2code.localization.validate_finetuned   --config configs/cassandra30.yaml   --epochs 1 2 3   --batch-size 128   --sanity-check-bugs 0


## 8. Validation — Component and Hybrid experiments


In [ ]:
!python -m bug2code.data.component_map   --config configs/cassandra30.yaml


In [ ]:
!python -m bug2code.localization.save_tfidf_val_scores   --config configs/cassandra30.yaml


In [ ]:
!python -m bug2code.localization.save_finetuned_val_scores   --config configs/cassandra30.yaml   --epoch 3


In [ ]:
!python -m bug2code.localization.component_experiment   --config configs/cassandra30.yaml   --epoch 3


In [ ]:
!python -m bug2code.localization.hybrid_experiment   --config configs/cassandra30.yaml   --epoch 3


## 9. Restore Epoch 3 for final Test evaluation

If the Colab runtime was restarted after training, restore the selected checkpoint from Drive.


In [ ]:
!mkdir -p experiments_cassandra30/finetuned_codebert/checkpoints

!cp "/content/drive/MyDrive/Bug2Code_transfer_local/epoch_3.pt"     experiments_cassandra30/finetuned_codebert/checkpoints/epoch_3.pt

!ls -lh experiments_cassandra30/finetuned_codebert/checkpoints/epoch_3.pt


## 10. Final Cassandra Test — generate scores


In [ ]:
!python -m bug2code.data.component_map   --config configs/cassandra30.yaml


In [ ]:
!python -m bug2code.localization.save_finetuned_test_scores   --config configs/cassandra30.yaml   --count-only


In [ ]:
!python -m bug2code.localization.save_tfidf_test_scores   --config configs/cassandra30.yaml


In [ ]:
!python -m bug2code.localization.save_finetuned_test_scores   --config configs/cassandra30.yaml


### Save raw Cassandra Test scores to Drive


In [ ]:
!mkdir -p "/content/drive/MyDrive/Bug2Code_results"

!cp "/content/Bug2Code/reports_cassandra30/tables/finetuned_epoch3_test_candidate_scores.parquet"     "/content/drive/MyDrive/Bug2Code_results/"

!cp "/content/Bug2Code/reports_cassandra30/tables/tfidf_test_candidate_scores.parquet"     "/content/drive/MyDrive/Bug2Code_results/"


## 11. Final Cassandra Test — Frozen CodeBERT


In [ ]:
!python -m bug2code.localization.frozen_codebert   --config configs/cassandra30.yaml   --split test 2>&1 | tee /content/drive/MyDrive/Bug2Code_results/frozen_codebert_test.log


In [ ]:
!cp /content/Bug2Code/reports_cassandra30/tables/frozen_codebert_test_results.csv    /content/drive/MyDrive/Bug2Code_results/


## 12. Final Cassandra Test — Hybrid


In [ ]:
!python -m bug2code.localization.hybrid_test_experiment   --config configs/cassandra30.yaml


In [ ]:
!cp /content/Bug2Code/reports_cassandra30/tables/hybrid_test_results.csv    /content/drive/MyDrive/Bug2Code_results/


## 13. Final Cassandra Test — Component filtering


In [ ]:
!python -m bug2code.localization.component_test_experiment   --config configs/cassandra30.yaml


In [ ]:
!cp /content/Bug2Code/reports_cassandra30/tables/component_test_results.csv    /content/drive/MyDrive/Bug2Code_results/

!cp /content/Bug2Code/reports_cassandra30/tables/component_test_per_bug.csv    /content/drive/MyDrive/Bug2Code_results/


## 14. Prepare HBase and Spark for cross-project evaluation

The same deterministic 30% sampling strategy is used for the HBase and Spark validation/test subsets.


In [ ]:
!python -m bug2code.data.dev_subset   --config configs/hbase30.yaml   --frac 0.3   --projects hbase   --splits val test


In [ ]:
!python -m bug2code.data.dev_subset   --config configs/spark30.yaml   --frac 0.3   --projects spark   --splits val test


In [ ]:
!python -m bug2code.data.repos --config configs/hbase30.yaml
!python -m bug2code.data.repos --config configs/spark30.yaml


In [ ]:
!python -m bug2code.localization.candidates --config configs/hbase30.yaml
!python -m bug2code.localization.candidates --config configs/spark30.yaml


## 15. Cross-project Test — count inference workload


In [ ]:
!python -m bug2code.localization.cross_project_scores   --config configs/hbase30.yaml   --split test   --count-only


In [ ]:
!python -m bug2code.localization.cross_project_scores   --config configs/spark30.yaml   --split test   --count-only


## 16. Cross-project Test — Spark

The selected Cassandra Epoch 3 checkpoint is evaluated directly on Spark.


In [ ]:
!python -m bug2code.localization.cross_project_scores   --config configs/spark30.yaml   --split test   --checkpoint /content/Bug2Code/experiments_cassandra30/finetuned_codebert/checkpoints/epoch_3.pt


### Download Spark raw candidate scores if needed


In [ ]:
from google.colab import files

files.download(
    "/content/Bug2Code/reports_spark30/tables/"
    "cassandra_epoch3_zeroshot_spark30_test_candidate_scores.parquet"
)


## 17. Cross-project Test — HBase

The same Cassandra checkpoint is evaluated directly on HBase.


In [ ]:
!python -m bug2code.localization.cross_project_scores   --config configs/hbase30.yaml   --split test   --checkpoint /content/Bug2Code/experiments_cassandra30/finetuned_codebert/checkpoints/epoch_3.pt


### Download HBase raw candidate scores if needed


In [ ]:
from google.colab import files

files.download(
    "/content/Bug2Code/reports_hbase30/tables/"
    "cassandra_epoch3_zeroshot_hbase30_test_candidate_scores.parquet"
)


## Notes

- Validation is used only for model/hyperparameter selection.
- Final reported comparisons are based on Test.
- Large checkpoints and raw candidate-score Parquet files are intentionally stored outside Git.
- The notebook records the Colab commands used for the final experiments; the project source code itself remains in the repository.
